In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "coping_capacity.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# FUNCTIONS
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5


# =============================================================================
# 1. DISTRICT-MONTH AGGREGATION
# =============================================================================
# Coping capacity variables:
# - HealthCenters (higher = better)
# - avg_electricity (higher = better)
# - block_piped_hhds_pct (higher = better access)

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          health_centers=("HealthCenters", "sum"),
          electricity=("avg_electricity", "mean"),
          piped_water=("block_piped_hhds_pct", "mean")
      )
)

# =============================================================================
# 2. MONTH-WISE Z-SCORES (ACROSS DISTRICTS)
# =============================================================================

vars_list = ["health_centers", "electricity", "piped_water"]

for var in vars_list:
    district_df[var + "_z"] = (
        district_df.groupby("timeperiod")[var]
        .transform(zscore)
    )

# =============================================================================
# 3. BINNING (1–5)
# =============================================================================

for var in vars_list:
    district_df[var + "_bin"] = district_df[var + "_z"].apply(classify)

# =============================================================================
# 4. COMPOSITE COPING CAPACITY SCORE
# =============================================================================

bin_cols = [v + "_bin" for v in vars_list]

district_df["coping_raw"] = district_df[bin_cols].sum(axis=1)

# =============================================================================
# 5. FINAL NORMALIZATION (MONTH-WISE)
# =============================================================================

district_df["coping_z"] = (
    district_df.groupby("timeperiod")["coping_raw"]
    .transform(zscore)
)

district_df["coping_capacity"] = district_df["coping_z"].apply(classify)

# =============================================================================
# 6. OUTPUT
# =============================================================================

output_cols = (
    ["district", "timeperiod"]
    + vars_list
    + [v + "_z" for v in vars_list]
    + bin_cols
    + ["coping_raw", "coping_z", "coping_capacity"]
)

district_df[output_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

# =============================================================================
# 7. CHECKS
# =============================================================================

print("\nCoping capacity distribution:")
print(district_df["coping_capacity"].value_counts().sort_index())

print("\nPreview:")
print(district_df.head())

Input shape: (7222, 20)
Saved: data/coping_capacity.csv

Coping capacity distribution:
coping_capacity
1     23
2    322
3    115
4    161
5     69
Name: count, dtype: int64

Preview:
  district timeperiod  health_centers  electricity  piped_water  \
0   Anugul    2023_01             209     9.846989     15.79471   
1   Anugul    2023_02             209     9.846989     15.79471   
2   Anugul    2023_03             209     9.846989     15.79471   
3   Anugul    2023_04             209     9.846989     15.79471   
4   Anugul    2023_05             209     9.846989     15.79471   

   health_centers_z  electricity_z  piped_water_z  health_centers_bin  \
0         -0.443077      -0.335803        0.93785                   3   
1         -0.443077      -0.335803        0.93785                   3   
2         -0.443077      -0.335803        0.93785                   3   
3         -0.443077      -0.335803        0.93785                   3   
4         -0.443077      -0.335803        0.9378